In [ ]:
"""
https://claude.ai/chat/630a6d73-29c4-4e4a-bf59-dbf49ba06517

stage0_events.py  --  Stage 0: event extraction for the MNQ JMA pivot intensity model.

RULES (traceability)
  S0.1  Sign of a series is sign(diff), with zero-runs carried forward from the last
        non-zero sign. Leading run before the first non-zero sign is 0 (masked).
  S0.2  A pivot (extremum) of series x at bar k is emitted as an EVENT at bar k+1,
        the first bar at which the turn is knowable. event_bar is the only time index
        used downstream. extremum_bar is retained for diagnostics only.
  S0.3  The target (MNQ JMA pivot) obeys S0.2 identically: target at bar t <=>
        jma_leg_dir[t] != jma_leg_dir[t-1]. Hence target_bar == event_bar and the
        target is causal at t.
  S0.4  Stage 2 features at bar t must use exciting events with event_bar <= t-1.
        This file does not enforce that; it only guarantees event_bar is knowable-at.
  S0.5  Polarity: +1 = local maximum at extremum_bar, -1 = local minimum.
  S0.6  jma_leg_dir at an event is read at event_bar (causal). opposing = polarity *
        jma_leg_dir; +1 = opposing the leg, -1 = confirming, 0 = leg dir undefined.
  S0.7  No cross-session spillage: sign, pivots, legs and leg_age are computed per
        session_date. First WARMUP_BARS bars of each session are masked from both
        event emission and target counting.
  S0.8  TICK is joined onto the MNQ bar grid with a backward asof on bar CLOSE
        (open + frame). Backward asof is causal. Step-held TICK values create flat
        runs; S0.1 zero-carry absorbs them.
  S0.9  Diagnostics report the recall ceiling: fraction of target events preceded by
        an opposing pivot of stream s within lag support L. This upper-bounds the
        recall of ANY causal model using stream s with support L.
"""

import argparse
import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- config

MNQ_COLS = {"timestamp": "timestamp", "jma": "jma", "d1": "jma_d1", "d2": "jma_d2"}
TICK_COLS = {"timestamp": "timestamp", "hma": "hma", "d1": "hma_d1", "d2": "hma_d2"}

TZ = "America/New_York"
WARMUP_BARS = 30
INCLUDE_TICK_HMA = False
ASOF_AT_BAR_CLOSE = False
LAG_SUPPORTS = [1, 2, 3, 5, 8, 13, 21, 34]


# ---------------------------------------------------------------- primitives

def signfill(x):
    """S0.1"""
    d = np.zeros(len(x), dtype=np.float64)
    d[1:] = np.diff(x)
    s = np.sign(d).astype(np.int8)
    pos = np.maximum.accumulate(np.where(s != 0, np.arange(len(s)), -1))
    out = np.zeros(len(s), dtype=np.int8)
    ok = pos >= 0
    out[ok] = s[pos[ok]]
    return out


def pivots(s):
    """S0.2, S0.5. s is a signfill array. Returns (event_bar, extremum_bar, polarity)."""
    if len(s) < 2:
        e = np.empty(0, dtype=np.int64)
        return e, e, np.empty(0, dtype=np.int8)
    turn = (s[1:] != s[:-1]) & (s[1:] != 0) & (s[:-1] != 0)
    event_bar = np.nonzero(turn)[0] + 1
    return event_bar, event_bar - 1, s[event_bar - 1]


# ---------------------------------------------------------------- load / align

def load(mnq_path, tick_path, frame_seconds):
    mnq = pd.read_parquet(mnq_path).rename(columns={v: k for k, v in MNQ_COLS.items()})
    mnq = mnq[["timestamp", "jma", "d1", "d2"]].copy()
    mnq["timestamp"] = pd.to_datetime(mnq["timestamp"], utc=True)
    mnq = mnq.sort_values("timestamp").reset_index(drop=True)

    tick = pd.read_parquet(tick_path).rename(columns={v: k for k, v in TICK_COLS.items()})
    tick = tick[["timestamp", "hma", "d1", "d2"]].copy()
    tick.columns = ["timestamp", "t_hma", "t_d1", "t_d2"]
    tick["timestamp"] = pd.to_datetime(tick["timestamp"], utc=True)
    tick = tick.sort_values("timestamp").reset_index(drop=True)

    key = mnq["timestamp"] + pd.Timedelta(seconds=frame_seconds) if ASOF_AT_BAR_CLOSE else mnq["timestamp"]
    left = mnq.assign(_k=key).sort_values("_k")
    df = pd.merge_asof(
        left, tick, left_on="_k", right_on="timestamp",
        direction="backward", tolerance=pd.Timedelta(seconds=max(2 * frame_seconds, 2)),
        suffixes=("", "_tick"),
    ).drop(columns=["_k", "timestamp_tick"])

    df["session_date"] = df["timestamp"].dt.tz_convert(TZ).dt.date
    df["bar_index"] = np.arange(len(df), dtype=np.int64)
    n0 = len(df)
    df = df.dropna(subset=["jma", "d1", "d2", "t_hma", "t_d1", "t_d2"]).reset_index(drop=True)
    return df, n0 - len(df)


# ---------------------------------------------------------------- extraction

def streams(df):
    spec = [("MNQ_D1", "d1"), ("MNQ_D2", "d2"), ("TICK_D1", "t_d1"), ("TICK_D2", "t_d2")]
    if INCLUDE_TICK_HMA:
        spec.append(("TICK_HMA", "t_hma"))
    return spec


def extract(df):
    ev_rows = []
    bar_frames = []
    spec = streams(df)

    for sess, g in df.groupby("session_date", sort=True):
        g = g.reset_index(drop=True)
        if len(g) <= WARMUP_BARS + 2:
            continue
        gbar = g["bar_index"].to_numpy()

        jma = g["jma"].to_numpy(np.float64)
        leg_dir = signfill(jma)                                    # S0.1
        #t_ev, t_ex, t_pol = pivots(leg_dir * 0 + signfill(jma))    # S0.3 (target)
        t_ev, t_ex, t_pol = pivots(signfill(jma))                 # S0.3 (targe

        keep = t_ev >= WARMUP_BARS                                 # S0.7
        t_ev, t_ex, t_pol = t_ev[keep], t_ex[keep], t_pol[keep]

        is_target = np.zeros(len(g), dtype=bool)
        is_target[t_ev] = True
        leg_id = np.cumsum(is_target)
        leg_start = np.zeros(len(g), dtype=np.int64)
        starts = np.concatenate(([0], t_ev))
        leg_start = starts[leg_id]
        leg_age = np.arange(len(g)) - leg_start
        leg_amp = np.abs(jma - jma[leg_start])

        bar_frames.append(pd.DataFrame({
            "bar_index": gbar,
            "timestamp": g["timestamp"].to_numpy(),
            "session_date": sess,
            "jma": jma,
            "jma_leg_dir": leg_dir,
            "leg_id": leg_id.astype(np.int32),
            "leg_age": leg_age.astype(np.int32),
            "leg_amp": leg_amp,
            "is_target": is_target,
            "warm": np.arange(len(g)) < WARMUP_BARS,
        }))

        for name, col in spec + [("MNQ_JMA_SELF", "jma")]:
            x = g[col].to_numpy(np.float64)
            e, k, pol = pivots(signfill(x))
            m = e >= WARMUP_BARS                                   # S0.7
            e, k, pol = e[m], k[m], pol[m]
            if len(e) == 0:
                continue
            xk = x[k]
            prev_e = np.concatenate(([-1], e[:-1]))
            prev_xk = np.concatenate(([np.nan], xk[:-1]))
            ev_rows.append(pd.DataFrame({
                "session_date": sess,
                "stream": name,
                "event_bar": gbar[e],
                "extremum_bar": gbar[k],
                "timestamp": g["timestamp"].to_numpy()[e],
                "polarity": pol,                                   # S0.5
                "jma_leg_dir": leg_dir[e],                         # S0.6
                "opposing": (pol * leg_dir[e]).astype(np.int8),    # S0.6
                "value": xk,
                "prev_leg_amp": np.abs(xk - prev_xk),
                "bars_since_prev": np.where(prev_e < 0, -1, e - prev_e).astype(np.int32),
                "leg_age_at_event": leg_age[e].astype(np.int32),
            }))

    bars = pd.concat(bar_frames, ignore_index=True)
    events = pd.concat(ev_rows, ignore_index=True).sort_values(["event_bar", "stream"]).reset_index(drop=True)
    events["stream_id"] = events["stream"].astype("category").cat.codes.astype(np.int8)
    return bars, events


# ---------------------------------------------------------------- diagnostics

def diagnostics(bars, events, dropped):
    tgt = np.sort(bars.loc[bars["is_target"], "bar_index"].to_numpy())
    n_bars = int((~bars["warm"]).sum())
    d = {
        "n_bars": int(len(bars)),
        "n_bars_ex_warmup": n_bars,
        "n_sessions": int(bars["session_date"].nunique()),
        "rows_dropped_on_join": int(dropped),
        "n_target": int(len(tgt)),
        "target_rate_per_1k": round(1000.0 * len(tgt) / max(n_bars, 1), 3),
        "median_leg_bars": float(np.median(np.diff(tgt))) if len(tgt) > 2 else None,
        "d1_sign_agrees_leg_dir": float(
            (np.sign(bars["d1"].to_numpy()) == bars["jma_leg_dir"].to_numpy()).mean()
        ) if "d1" in bars.columns else None,
        "streams": {},
        "recall_ceiling": {},
        "precision_base": {},
    }

    for name, g in events.groupby("stream"):
        eb = np.sort(g["event_bar"].to_numpy())
        d["streams"][name] = {
            "n": int(len(g)),
            "rate_per_1k": round(1000.0 * len(g) / max(n_bars, 1), 3),
            "ratio_to_target": round(len(g) / max(len(tgt), 1), 2),
            "median_inter_event_bars": float(np.median(np.diff(eb))) if len(eb) > 2 else None,
            "frac_opposing": float((g["opposing"] == 1).mean()),
        }

    opp = events[events["opposing"] == 1]
    for name, g in opp.groupby("stream"):                          # S0.9
        eb = np.sort(g["event_bar"].to_numpy())
        rec, prec = {}, {}
        for L in LAG_SUPPORTS:
            lo = np.searchsorted(eb, tgt - L, side="left")
            hi = np.searchsorted(eb, tgt, side="left")
            rec[L] = round(float((hi > lo).mean()), 4)
            a = np.searchsorted(tgt, eb + 1, side="left")
            b = np.searchsorted(tgt, eb + L, side="right")
            prec[L] = round(float((b > a).mean()), 4)
        d["recall_ceiling"][name] = rec
        d["precision_base"][name] = prec

    eb = np.sort(opp["event_bar"].unique())
    d["recall_ceiling"]["ANY_OPPOSING"] = {
        L: round(float((np.searchsorted(eb, tgt, "left") > np.searchsorted(eb, tgt - L, "left")).mean()), 4)
        for L in LAG_SUPPORTS
    }
    return d


# ---------------------------------------------------------------- main

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--mnq", required=True)
    p.add_argument("--tick", required=True)
    p.add_argument("--frame", type=int, required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--assume-aligned", action="store_true")
    a = p.parse_args()

    if a.assume_aligned:
        global ASOF_AT_BAR_CLOSE
        ASOF_AT_BAR_CLOSE = False

    df, dropped = load(a.mnq, a.tick, a.frame)
    bars, events = extract(df)
    bars = bars.merge(df[["bar_index", "d1"]], on="bar_index", how="left")
    diag = diagnostics(bars, events, dropped)

    bars.to_parquet(f"{a.out}/bars_{a.frame}s.parquet", index=False)
    events.to_parquet(f"{a.out}/events_{a.frame}s.parquet", index=False)
    with open(f"{a.out}/stage0_diagnostics_{a.frame}s.json", "w") as f:
        json.dump(diag, f, indent=2, default=str)
    print(json.dumps(diag, indent=2, default=str))


if __name__ == "__main__":
    main()